In [1]:
import os, chromadb
from ollama import generate 
from PIL import Image

def get_vlm_description(image_path, model_name):
    """利用 VLM 分析圖片內容"""
    prompt = "請詳細描述這張花卉照片的特徵，包括花種、顏色、外觀細節以及適合的送禮場合。請用繁體中文回答。"
    response = generate(model=model_name, prompt=prompt, images=[image_path])
    return response['response']

def build_index_with_vlm(IMAGE_FOLDER, db_path , collection_name, model_name):
    client = chromadb.PersistentClient(path=db_path)
    collection = client.get_or_create_collection(name=collection_name)
    image_files = [f for f in os.listdir(IMAGE_FOLDER) if f.endswith(('.jpg', '.png'))]
    
    for filename in image_files:
        path = os.path.join(IMAGE_FOLDER, filename)        
        # 1. 提取檔名基本資訊
        flower_name = filename.split('_')[0]        
        # 2. 透過 VLM 獲取深度特徵 (這就是檢索的依據)
        print(f"正在分析 {filename}...")
        vlm_description = get_vlm_description(path, model_name)        
        # 3. 存入 ChromaDB
        # 將「自動生成的描述」與「檔名」結合作為 document
        combined_text = f"品種:{flower_name}。特徵分析:{vlm_description}"        
        collection.add(
            documents=[combined_text],
            metadatas=[{"flower_type": flower_name, "filename": filename}],
            ids=[filename]
        )
    print(f"分析完成")

if __name__ == "__main__":
    build_index_with_vlm(IMAGE_FOLDER = "./img_db", db_path="./chroma_flower_db" , 
                         collection_name="flowers", model_name="qwen3-vl:30B-A3B-Thinking")

正在分析 向日葵_1.jpg...
正在分析 向日葵_2.jpg...
正在分析 向日葵_3.jpg...
正在分析 向日葵_4.jpg...
正在分析 百合_1.jpg...
正在分析 百合_2.jpg...
正在分析 百合_3.jpg...
正在分析 百合_4.jpg...
分析完成
